<a href="https://colab.research.google.com/github/marciapaulasv-coder/TechChalenge/blob/main/02_preprocessamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn import*

df = pd.read_csv('base_modelagem.csv')

df.shape

(36457, 22)

In [14]:
# Verificação de IDs duplicados
print('Quantidade de registros:', len(df))
print('Quantidade de IDs únicos:', df['ID'].nunique())
print('Quantidade de IDs duplicados:', df['ID'].duplicated().sum())

Quantidade de registros: 36457
Quantidade de IDs únicos: 36457
Quantidade de IDs duplicados: 0


**Conclusão:** A base de modelagem possui 36.457 registros e 36.457 IDs únicos, não sendo identificados clientes duplicados.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   36457 non-null  int64  
 1   CODE_GENDER          36457 non-null  int64  
 2   FLAG_OWN_CAR         36457 non-null  int64  
 3   FLAG_OWN_REALTY      36457 non-null  int64  
 4   CNT_CHILDREN         36457 non-null  int64  
 5   AMT_INCOME_TOTAL     36457 non-null  float64
 6   NAME_INCOME_TYPE     36457 non-null  object 
 7   NAME_EDUCATION_TYPE  36457 non-null  object 
 8   NAME_FAMILY_STATUS   36457 non-null  object 
 9   NAME_HOUSING_TYPE    36457 non-null  object 
 10  DAYS_BIRTH           36457 non-null  int64  
 11  DAYS_EMPLOYED        36457 non-null  int64  
 12  FLAG_MOBIL           36457 non-null  int64  
 13  FLAG_WORK_PHONE      36457 non-null  int64  
 14  FLAG_PHONE           36457 non-null  int64  
 15  FLAG_EMAIL           36457 non-null 

In [15]:
df_modelo = df.drop(columns=['ID', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'PERFIL'])

df_modelo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CODE_GENDER          36457 non-null  int64  
 1   FLAG_OWN_CAR         36457 non-null  int64  
 2   FLAG_OWN_REALTY      36457 non-null  int64  
 3   CNT_CHILDREN         36457 non-null  int64  
 4   AMT_INCOME_TOTAL     36457 non-null  float64
 5   NAME_INCOME_TYPE     36457 non-null  object 
 6   NAME_EDUCATION_TYPE  36457 non-null  object 
 7   NAME_FAMILY_STATUS   36457 non-null  object 
 8   NAME_HOUSING_TYPE    36457 non-null  object 
 9   FLAG_MOBIL           36457 non-null  int64  
 10  FLAG_WORK_PHONE      36457 non-null  int64  
 11  FLAG_PHONE           36457 non-null  int64  
 12  FLAG_EMAIL           36457 non-null  int64  
 13  OCCUPATION_TYPE      25134 non-null  object 
 14  CNT_FAM_MEMBERS      36457 non-null  float64
 15  TARGET               36457 non-null 

**Conclusão:** foram excluídas as colunas ID, por representar apenas a identificação do cliente, e PERFIL, por representar a mesma informação do TARGET. Também foram excluídas DAYS_BIRTH e DAYS_EMPLOYED, pois suas informações já estão representadas pelas variáveis IDADE e ANOS_EMPREGADO, respectivamente.

**Tratamento coluna "OCCUPATION_TYPE"**

In [5]:
df_modelo['OCCUPATION_TYPE'].value_counts(dropna=False)

,count
OCCUPATION_TYPE,
NaN,11323
Laborers,6211
Core staff,3591
Sales staff,3485
Managers,3012
Drivers,2138
High skill tech staff,1383
Accountants,1241
Medicine staff,1207


In [6]:
df_modelo['OCCUPATION_TYPE'] = df_modelo['OCCUPATION_TYPE'].fillna('Não informado')
df_modelo['OCCUPATION_TYPE'].value_counts()

,count
OCCUPATION_TYPE,
Não informado,11323
Laborers,6211
Core staff,3591
Sales staff,3485
Managers,3012
Drivers,2138
High skill tech staff,1383
Accountants,1241
Medicine staff,1207


**Conclusão:** A variável OCCUPATION_TYPE apresentou 11.323 valores ausentes, ou seja 31% da base. Como a quantia é muito significativa, optou-se por não excluir a informação e foi criada a categoria “Não informado”.

**Tratamento coluna "ANOS_EMPREGADO"**

In [7]:
df[df['ANOS_EMPREGADO'].isna()]['NAME_INCOME_TYPE'].value_counts(dropna=False)

,count
NAME_INCOME_TYPE,
Pensioner,6135


In [8]:
df_modelo['ANOS_EMPREGADO'] = df_modelo['ANOS_EMPREGADO'].fillna(0)
df_modelo['ANOS_EMPREGADO'].isna().sum()

np.int64(0)

**Conclusão:** Os valores ausentes dessa variável correspondem aos clientes pensionistas e foram gerados durante a transformação da variável original DAYS_EMPLOYED, que apresentava o código especial 365243. Para esses casos, foi atribuído o valor 0. Entendendo-se que por não estarem trabalhando, não tem tempo de emprego.

In [9]:
df_modelo.isna().sum()

,0
CODE_GENDER,0
FLAG_OWN_CAR,0
FLAG_OWN_REALTY,0
CNT_CHILDREN,0
AMT_INCOME_TOTAL,0
NAME_INCOME_TYPE,0
NAME_EDUCATION_TYPE,0
NAME_FAMILY_STATUS,0
NAME_HOUSING_TYPE,0
FLAG_MOBIL,0


In [10]:
colunas_categoricas = df_modelo.select_dtypes(include='object').columns

for coluna in colunas_categoricas:
    print(f'\n{coluna}')
    print(df_modelo[coluna].value_counts())


NAME_INCOME_TYPE
NAME_INCOME_TYPE
Working                 18819
Commercial associate     8490
Pensioner                6152
State servant            2985
Student                    11
Name: count, dtype: int64

NAME_EDUCATION_TYPE
NAME_EDUCATION_TYPE
Secondary / secondary special    24777
Higher education                  9864
Incomplete higher                 1410
Lower secondary                    374
Academic degree                     32
Name: count, dtype: int64

NAME_FAMILY_STATUS
NAME_FAMILY_STATUS
Married                 25048
Single / not married     4829
Civil marriage           2945
Separated                2103
Widow                    1532
Name: count, dtype: int64

NAME_HOUSING_TYPE
NAME_HOUSING_TYPE
House / apartment      32548
With parents            1776
Municipal apartment     1128
Rented apartment         575
Office apartment         262
Co-op apartment          168
Name: count, dtype: int64

OCCUPATION_TYPE
OCCUPATION_TYPE
Não informado            11323
Laborers   

In [11]:
df_modelo = pd.get_dummies(
    df_modelo,
    columns=colunas_categoricas,
    drop_first=True,
    dtype=int
)

In [13]:
df['ID'].duplicated().sum()
df['ID'].nunique()

36457